In [ ]:
# In[1]:

import pandas as pd
import pytz

# Load CSVs into DataFrames and keep them in kernel variables
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_logs_df = pd.read_csv("error_logs.csv")

# Helper: convert timestamp column (seconds) to timezone-aware datetime in Asia/Shanghai (UTC+8)
tz = pytz.timezone("Asia/Shanghai")
def to_tz_datetime(df, ts_col="timestamp"):
    # Convert to datetime (unit seconds) then localize to Asia/Shanghai
    # If already datetime, leave as is
    dt = pd.to_datetime(df[ts_col], unit="s")
    # Localize naive timestamps to Asia/Shanghai
    dt = dt.dt.tz_localize(tz)
    return dt

# Add timezone-aware datetime columns for each DataFrame (kept in variables)
metric_df["dt"] = to_tz_datetime(metric_df, "timestamp")
trace_df["dt"] = to_tz_datetime(trace_df, "timestamp")
log_df["dt"] = to_tz_datetime(log_df, "timestamp")
error_logs_df["dt"] = to_tz_datetime(error_logs_df, "timestamp")

# Helper to create compact overview: total rows, min ts, max ts, top 20 cmdb_id counts
def compact_overview(df, name):
    total_rows = df.shape[0]
    if total_rows == 0:
        min_ts = pd.NaT
        max_ts = pd.NaT
        top_cmdb = pd.DataFrame(columns=["cmdb_id", "count"])
    else:
        min_ts = df["dt"].min()
        max_ts = df["dt"].max()
        top_cmdb = df.groupby("cmdb_id").size().reset_index(name="count").sort_values("count", ascending=False).head(20)
    overview = pd.DataFrame({
        "source": [name],
        "rows": [total_rows],
        "min_timestamp": [min_ts],
        "max_timestamp": [max_ts]
    })
    return overview, top_cmdb

# Create summaries for each file
metric_summary_df, metric_top_cmdb_df = compact_overview(metric_df, "metric.csv")
trace_summary_df, trace_top_cmdb_df = compact_overview(trace_df, "trace.csv")
log_summary_df, log_top_cmdb_df = compact_overview(log_df, "log.csv")
error_summary_df, error_top_cmdb_df = compact_overview(error_logs_df, "error_logs.csv")

# Additional requested lists and top-k for metric, trace, log
# Metric: unique kpi_name values (show up to 20) and top 20 kpi_name by count
if "kpi_name" in metric_df.columns:
    metric_kpi_counts_df = metric_df.groupby("kpi_name").size().reset_index(name="count").sort_values("count", ascending=False).head(20)
    metric_kpi_unique_df = pd.DataFrame({"kpi_name": metric_df["kpi_name"].unique()[:20]})
else:
    metric_kpi_counts_df = pd.DataFrame(columns=["kpi_name", "count"])
    metric_kpi_unique_df = pd.DataFrame(columns=["kpi_name"])

# Trace: unique trace_name values and top 20 trace_name by count
if "trace_name" in trace_df.columns:
    trace_trace_counts_df = trace_df.groupby("trace_name").size().reset_index(name="count").sort_values("count", ascending=False).head(20)
    trace_trace_unique_df = pd.DataFrame({"trace_name": trace_df["trace_name"].unique()[:20]})
else:
    trace_trace_counts_df = pd.DataFrame(columns=["trace_name", "count"])
    trace_trace_unique_df = pd.DataFrame(columns=["trace_name"])

# Log: unique log_name values and top 20 log_name by count
if "log_name" in log_df.columns:
    log_log_counts_df = log_df.groupby("log_name").size().reset_index(name="count").sort_values("count", ascending=False).head(20)
    log_log_unique_df = pd.DataFrame({"log_name": log_df["log_name"].unique()[:20]})
else:
    log_log_counts_df = pd.DataFrame(columns=["log_name", "count"])
    log_log_unique_df = pd.DataFrame(columns=["log_name"])

# Error logs: show up to top 20 rows with timestamp, cmdb_id, and first 200 characters of message
if "message" in error_logs_df.columns:
    error_logs_top20_df = error_logs_df.loc[:, ["timestamp", "cmdb_id", "message", "dt"]].head(20).copy()
    # Truncate message to first 200 characters
    error_logs_top20_df["message_snippet"] = error_logs_top20_df["message"].astype(str).str.slice(0, 200)
    # Keep requested columns only
    error_logs_top20_df = error_logs_top20_df.loc[:, ["timestamp", "cmdb_id", "message_snippet", "dt"]]
else:
    error_logs_top20_df = pd.DataFrame(columns=["timestamp", "cmdb_id", "message_snippet", "dt"])

# Keep output compact: only show the created summary tables (top 20 lists where applicable)
metric_summary_df, metric_top_cmdb_df, metric_kpi_unique_df, metric_kpi_counts_df, \
trace_summary_df, trace_top_cmdb_df, trace_trace_unique_df, trace_trace_counts_df, \
log_summary_df, log_top_cmdb_df, log_log_unique_df, log_log_counts_df, \
error_summary_df, error_top_cmdb_df, error_logs_top20_df

```
Out[1]:
```
# Concise plain-English summary based on the loaded telemetry (Asia/Shanghai timezone)
summary = (
    "Summary of loaded telemetry (timestamps shown in Asia/Shanghai UTC+8):\n\n"
    "metric.csv:\n"
    "- Total rows: 322,439\n"
    "- Timestamp range: 2022-03-21 01:00:00+08:00 to 2022-03-21 02:00:00+08:00\n"
    "- Top cmdb_id by row count (top examples): adservice (20,801), adservice2 (20,313), "
    "frontend2-0 (12,200), frontend-2/1/0 (~11,468 each), checkoutservice2-0 (9,028), "
    "checkoutservice-1/2/0 (~8,784 / 8,540 / 8,540), several productcatalogservice and recommendationservice pods (7,000–6,300).\n"
    "- KPI inventory: many unique kpi_name values (runtime, app.grpc/app.http, container.*, etc.).\n"
    "- Top kpi_name examples by count: a group of container.node-6.* metrics appear among the most frequent (counts ~2,196 and ~2,074 for various container CPU/memory/network/fs metrics).\n\n"
    "trace.csv:\n"
    "- Total rows: 34,344\n"
    "- Timestamp range: 2022-03-21 01:30:00+08:00 to 2022-03-21 01:59:00+08:00\n"
    "- Top cmdb_id by row count (top examples): frontend-0 (2,716), frontend-1 (2,684), frontend-2 (2,572), checkoutservice-0/2/1 (~2,540 / 2,428 / 2,392), productcatalogservice pods (~1,180–1,192), several recommendationservice pods (~840).\n"
    "- Trace metrics: per-source features like duration_mean, duration_p95, error_rate, row_count. Top trace_name examples: trace.from_frontend-0.* (each ~619), similar groups for frontend-1 and frontend-2; checkoutservice traces also frequent (~504 / 483).\n\n"
    "log.csv:\n"
    "- Total rows: 1,834\n"
    "- Timestamp range: 2022-03-21 01:30:00+08:00 to 2022-03-21 02:00:00+08:00\n"
    "- Top cmdb_id examples: many service pods with ~60–62 rows each (shows logs fairly sparse and evenly distributed across pods).\n"
    "- Unique log_name values: log.error_count and log.row_count (each appears 917 times).\n\n"
    "error_logs.csv:\n"
    "- Total rows: 0 (empty)\n\n"
    "Notes:\n"
    "- Full DataFrames are stored in kernel variables: metric_df, trace_df, log_df, error_logs_df for any further analysis.\n"
    "- metric and trace telemetry are dense and cover many services/pods; log telemetry is sparse; there are no raw error log entries in error_logs.csv for this time slice."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(       source    rows             min_timestamp             max_timestamp
0  metric.csv  322439 2022-03-21 01:00:00+08:00 2022-03-21 02:00:00+08:00,                      cmdb_id  count
0                  adservice  20801
4                 adservice2  20313
30               frontend2-0  12200
29                frontend-2  11468
28                frontend-1  11468
27                frontend-0  11468
15        checkoutservice2-0   9028
13         checkoutservice-1   8784
14         checkoutservice-2   8540
12         checkoutservice-0   8540
44   productcatalogservice-1   7320
43   productcatalogservice-0   7320
45   productcatalogservice-2   7076
46  productcatalogservice2-0   6344
5               adservice2-0   6222
51  recommendationservice2-0   6222
10            cartservice2-0   6100
9              cartservice-2   6100
7              cartservice-0   6100
50   recommendationservice-2   5978,                                              kpi_name
0                                      app.grpc.count
1                                        app.grpc.mrt
2                                         app.grpc.rr
3                                         app.grpc.sr
4                                      app.http.count
5                                        app.http.mrt
6                                         app.http.rr
7                                         app.http.sr
8     runtime.java_lang_ClassLoading_LoadedClassCount
9   runtime.java_lang_ClassLoading_TotalLoadedClas...
10  runtime.java_lang_ClassLoading_UnloadedClassCount
11             runtime.java_lang_ClassLoading_Verbose
12  runtime.java_lang_Compilation_CompilationTimeM...
13  runtime.java_lang_Compilation_TotalCompilation...
14  runtime.java_lang_GarbageCollector_CollectionC...
15  runtime.java_lang_GarbageCollector_CollectionC...
16  runtime.java_lang_GarbageCollector_CollectionT...
17  runtime.java_lang_GarbageCollector_CollectionT...
18  runtime.java_lang_GarbageCollector_LastGcInfo_...
19  runtime.java_lang_GarbageCollector_LastGcInfo_...,                                               kpi_name  count
119  container.node-6.container_network_transmit_pa...   2196
118  container.node-6.container_network_transmit_pa...   2196
112  container.node-6.container_network_receive_MB....   2196
113  container.node-6.container_network_receive_err...   2196
117  container.node-6.container_network_transmit_er...   2196
116  container.node-6.container_network_transmit_MB...   2196
115  container.node-6.container_network_receive_pac...   2196
114  container.node-6.container_network_receive_pac...   2196
126  container.node-6.container_spec_memory_swap_li...   2074
78         container.node-6.container_cpu_user_seconds   2074
77        container.node-6.container_cpu_usage_seconds   2074
76       container.node-6.container_cpu_system_seconds   2074
75     container.node-6.container_cpu_load_average_10s   2074
74   container.node-6.container_cpu_cfs_throttled_s...   2074
110         container.node-6.container_memory_usage_MB   2074
111   container.node-6.container_memory_working_set_MB   2074
96      container.node-6.container_fs_writes./dev/vda1   2074
97    container.node-6.container_fs_writes_MB./dev/vda   2074
104  container.node-6.container_memory_failures.hie...   2074
105  container.node-6.container_memory_failures.hie...   2074,       source   rows             min_timestamp             max_timestamp
0  trace.csv  34344 2022-03-21 01:30:00+08:00 2022-03-21 01:59:00+08:00,                     cmdb_id  count
20               frontend-0   2716
21               frontend-1   2684
22               frontend-2   2572
8         checkoutservice-0   2540
10        checkoutservice-2   2428
9         checkoutservice-1   2392
30  productcatalogservice-2   1192
28  productcatalogservice-0   1180
29  productcatalogservice-1   1180
23              frontend2-0   1024
32  recommendationservice-0    840
33  recommendationservice-1    840
34  recommendationservice-2    840
4             cartservice-0    832
5             cartservice-1    824
6             cartservice-2    824
14        currencyservice-2    720
12        currencyservice-0    716
13        currencyservice-1    716
36        shippingservice-0    712,                                     trace_name
0          trace.from_frontend-0.duration_mean
1           trace.from_frontend-0.duration_p95
2             trace.from_frontend-0.error_rate
3              trace.from_frontend-0.row_count
4          trace.from_frontend-1.duration_mean
5           trace.from_frontend-1.duration_p95
6             trace.from_frontend-1.error_rate
7              trace.from_frontend-1.row_count
8          trace.from_frontend-2.duration_mean
9           trace.from_frontend-2.duration_p95
10            trace.from_frontend-2.error_rate
11             trace.from_frontend-2.row_count
12        trace.from_frontend2-0.duration_mean
13         trace.from_frontend2-0.duration_p95
14           trace.from_frontend2-0.error_rate
15            trace.from_frontend2-0.row_count
16  trace.from_checkoutservice-0.duration_mean
17   trace.from_checkoutservice-0.duration_p95
18     trace.from_checkoutservice-0.error_rate
19      trace.from_checkoutservice-0.row_count,                                     trace_name  count
16         trace.from_frontend-0.duration_mean    619
17          trace.from_frontend-0.duration_p95    619
18            trace.from_frontend-0.error_rate    619
19             trace.from_frontend-0.row_count    619
20         trace.from_frontend-1.duration_mean    611
21          trace.from_frontend-1.duration_p95    611
22            trace.from_frontend-1.error_rate    611
23             trace.from_frontend-1.row_count    611
27             trace.from_frontend-2.row_count    583
26            trace.from_frontend-2.error_rate    583
25          trace.from_frontend-2.duration_p95    583
24         trace.from_frontend-2.duration_mean    583
3       trace.from_checkoutservice-0.row_count    504
2      trace.from_checkoutservice-0.error_rate    504
1    trace.from_checkoutservice-0.duration_p95    504
0   trace.from_checkoutservice-0.duration_mean    504
11      trace.from_checkoutservice-2.row_count    483
10     trace.from_checkoutservice-2.error_rate    483
9    trace.from_checkoutservice-2.duration_p95    483
8   trace.from_checkoutservice-2.duration_mean    483,     source  rows             min_timestamp             max_timestamp
0  log.csv  1834 2022-03-21 01:30:00+08:00 2022-03-21 02:00:00+08:00,                     cmdb_id  count
1               adservice-1     62
5             cartservice-2     62
4             cartservice-1     62
9         currencyservice-0     62
11        currencyservice-2     62
10        currencyservice-1     62
15               frontend-0     62
23  productcatalogservice-2     62
24  recommendationservice-0     62
17               frontend-2     62
25  recommendationservice-1     62
6         checkoutservice-0     60
3             cartservice-0     60
0               adservice-0     60
2               adservice-2     60
22  productcatalogservice-1     60
14           emailservice-2     60
13           emailservice-1     60
7         checkoutservice-1     60
8         checkoutservice-2     60,           log_name
0  log.error_count
1    log.row_count,           log_name  count
0  log.error_count    917
1    log.row_count    917,            source  rows min_timestamp max_timestamp
0  error_logs.csv     0           NaT           NaT, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message_snippet, dt]
Index: [])```
```